# MuniciPAL: Data Ingestion & LLM Engine
**Course:** CAP 6951 - Graduate Project  
**Team:** Team 13  
**Lead:** Allison Cerna  
**Last Updated:** July 6th, 2026

---

## Project Objective
This notebook serves as the primary workspace for the **MuniciPAL** RAG pipeline. 
- **Goal:** Ingest municipal ordinances and grant policies to provide grounded, accurate AI-assisted responses.
- **Current Task:** Automated data manifest generation and LLM response testing.

## Notebook Structure
1. **Environment Setup:** Importing dependencies.
2. **Data Ingestion:** Running the automated scanner for PDF documents.
3. **LLM Engine:** Prototyping the grounded response logic.
4. **Validation:** Running tests against our gold-standard dataset.

---

In [ ]:
# Installing required libraries directly from the notebook
%pip install pandas openai pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install -q -U google-generativeai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
%pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Clearing any potentially stale environment variables that might conflict
for key in list(os.environ.keys()):
    if key.startswith("GOOGLE_"):
        del os.environ[key]

# Forcing reload from the .env file in the current directory
load_dotenv(dotenv_path=".env", override=True)

# Verifying key now
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("CRITICAL ERROR: GOOGLE_API_KEY is not found. Check your .env file path.")
else:
    print("API Key loaded successfully.")
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.0-flash')
    
    try:
        response = model.generate_content("Say 'Test successful'")
        print(f"Response: {response.text}")
    except Exception as e:
        print(f"Final connection attempt failed with: {e}")

c:\Users\allis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\allis\AppData\Local\Temp\ipykernel_25252\4066838919.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


API Key loaded successfully.
Final connection attempt failed with: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 49.291655403s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: 

In [2]:
import google.generativeai as genai

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [3]:
# ==============================================================================
# INGESTION: Extracting text from our nested PDF structure
# By: Allison Cerna (Team 13 Lead)
# ==============================================================================

import pdfplumber
import pandas as pd
import os

def extract_text_from_pdf(pdf_path):
    # Grabbing the text from every page of the PDF.
    # Since these are text-selectable, this should be super clean.
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Bummer, couldn't read {pdf_path}: {e}")
    return text

def process_pdfs_to_manifest(root_folder='data/raw', output_csv='data/manifest.csv'):
    all_data = []
    
    # Using os.walk to go through all subfolders (like the year folders in ordinances).
    # This ensures we hit every single PDF regardless of how deep it's buried.
    for root, dirs, files in os.walk(root_folder):
        for filename in files:
            if filename.endswith(".pdf"):
                path = os.path.join(root, filename)
                print(f"Processing: {filename}...")
                content = extract_text_from_pdf(path)
                
                # Adding it to our list so we can turn it into a CSV later.
                all_data.append({"filename": filename, "text": content})
    
    # Saving everything into a CSV so the rest of our pipeline can access it.
    df = pd.DataFrame(all_data)
    df.to_csv(output_csv, index=False)
    print(f"Done! Processed {len(all_data)} PDFs into {output_csv}.")

if __name__ == "__main__":
    # Pointing it at the raw data folder to start the ingestion.
    process_pdfs_to_manifest()

Processing: Accounts Receivable (BF-26, Rev. 1).pdf...
Processing: Budget Transfer and Amendment Policy (BF-7, Rev. 8).pdf...
Processing: Grant Administration (BF-24, Rev. 2).pdf...
Processing: Ordinance No. 01-24.pdf...
Processing: Ordinance No. 03-24.pdf...
Processing: Ordinance No. 04-24.pdf...
Processing: Ordinance No. 05-24.pdf...
Processing: Ordinance No. 06-24.pdf...
Processing: Ordinance No. 08-24.pdf...
Processing: Ordinance No. 11-24.pdf...
Processing: Ordinance No. 12-24.pdf...
Processing: Ordinance No. 13-24.pdf...
Processing: Ordinance No. 14-24.pdf...
Processing: Ordinance No. 18-24.pdf...
Processing: Ordinance No. 20-24.pdf...
Processing: Ordinance No. 21-24.pdf...
Processing: Ordinance No. 22-24.pdf...
Processing: Ordinance No. 23-24.pdf...
Processing: Ordinance No. 24-24.pdf...
Processing: Ordinance No. 26-24.pdf...
Processing: Ordinance No. 28-24.pdf...
Processing: Ordinance No. 32-24.pdf...
Processing: Ordinance No. 33-24.pdf...
Processing: Ordinance No. 35-24.pdf...

In [5]:
# ==============================================================================
# LLM INTEGRATION: MuniciPAL (Google Gemini Version)
# By: Allison Cerna (Team 13 Lead)
# ==============================================================================

import os
import google.generativeai as genai

class MuniciPALEngine:
    def __init__(self, api_key=None):
        # We're passing the key in so we can swap between real/mock modes easily.
        self.api_key = api_key
        if self.api_key:
            # Initialize Google Generative AI with your API key
            genai.configure(api_key=self.api_key)
            # Using Gemini 1.5 Flash as it is efficient and free for your use case
            self.model = genai.GenerativeModel('gemini-1.5-flash')
        else:
            self.model = None

    def generate_response(self, user_query, retrieved_chunks):
        # Mock Mode for pipeline testing
        if not self.model:
            return f"[MOCK MODE] Logic test: Successfully processed query '{user_query}' with {len(retrieved_chunks)} chunks."
        
        # Joins our context chunks into one clean string
        context = "\n\n".join(retrieved_chunks)
        
        # Grounding the model with a system instruction
        system_instruction = (
            "You are a municipal policy assistant for Team 13 (MuniciPAL). "
            "Use ONLY the provided context to answer the user's question. "
            "If the answer is not found in the context, state: "
            "'I do not have enough information in the municipal records to answer this.'"
        )
        
        # Prompt construction
        prompt = f"{system_instruction}\n\nContext: {context}\n\nQuestion: {user_query}"
        
        # Generates the response using Gemini
        response = self.model.generate_content(prompt)
        
        # Returns the generated text
        return response.text

In [6]:
import pandas as pd

def get_relevant_chunks(query, manifest_path='data/manifest.csv'):
    # This is our skeleton retrieval logic. 
    # For now, it just returns a list of dummy chunks so we can test the 
    # full pipeline end-to-end. Matthew can replace this later with 
    # actual ChromaDB search logic once he's ready.
    
    print(f"Retrieving relevant chunks for query: '{query}'...")
    
    # Placeholder: Returning a hardcoded list of text chunks. 
    # This ensures the pipeline is wired up and working while we wait 
    # for the real vector database integration.
    return [
        "Ordinance 2026-01: All grant applications must be submitted by June 1st.",
        "Policy BF-7: Budget transfers require approval from the City Manager."
    ]

# Testing integration/llm

In [16]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#trying to debug api key issues, not connecting
from dotenv import load_dotenv
import os

load_dotenv()

print(f"Key loaded: {'Yes' if os.getenv('GOOGLE_API_KEY') else 'No'}")

Key loaded: Yes


In [ ]:
from src.llm_engine import MuniciPALEngine
import os

test_engine = MuniciPALEngine(api_key=os.getenv("GOOGLE_API_KEY"))

try:
    print("Testing direct engine call...")
    # Testing to see if it crashes, trying to debug api issues
    response = test_engine.generate_response("Say hello", ["Hello world"])
    print("Success:", response)
except Exception as e:
    print("Engine crashed independently:", repr(e))

Testing direct engine call...
Engine crashed independently: ResourceExhausted('You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 58.362067379s.')


In [ ]:
# ==============================================================================
# Orchestrator
# By: Allison Cerna (Team 13 Lead)
# ==============================================================================
import pandas as pd
from src.llm_engine import MuniciPALEngine
import os

# Initializing the engine. Using the API key here to keep us flexible
engine = MuniciPALEngine(api_key=os.getenv("GOOGLE_API_KEY"))

def retrieve_relevant_chunks(query, csv_path='data/manifest.csv'):
    """
    Temporary placeholder for the retrieval layer.
    Keyword check against our manifest.csv for now; this *must* be 
    swapped out for the ChromaDB index once Matthew finishes it.
    """
    df = pd.read_csv(csv_path)
    # Basic filtering to grab potentially relevant content for the query
    results = df[df['text'].str.contains(query, case=False, na=False)]
    
    # Returning top match
    return results['text'].head(1).tolist()

def run_pipeline(user_query):
    # Breaking the query into keywords to improve our retrieval rate
    # Ignoring short words to reduce noise in the search results
    keywords = [word for word in user_query.split() if len(word) > 3]
    
    # Searching for any match in our manifest
    df = pd.read_csv('data/manifest.csv')
    
    # Filtering rows that contain at least one keyword
    mask = df['text'].apply(lambda x: any(k.lower() in str(x).lower() for k in keywords))
    results = df[mask]
    
    # Limiting 
    chunks = results['text'].head(1).tolist()
    
    if not chunks:
        return "I do not have enough information."
    
    # Final pass through the engine to get a grounded response
    return engine.generate_response(user_query, chunks)

In [23]:
import time
# test_pipeline.py
from src.orchestrator import run_pipeline

# Choosing a query that we know should be in our PDF documents
query = "What is the policy regarding municipal grant deadlines?"

print(f"Testing pipeline with query: '{query}'...")

# This calls our orchestrator, which triggers retrieval and generation
time.sleep(60)
response = run_pipeline(query)

print("-" * 30)
print(f"Assistant Response:\n{response}")
print("-" * 30)

Testing pipeline with query: 'What is the policy regarding municipal grant deadlines?'...


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 52.146363349s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quo

In [19]:
import os
import pandas as pd
import pdfplumber
import google.generativeai as genai
import sys

sys.path.append(os.getcwd())

# API key
os.environ["GOOGLE_API_KEY"] = "YOUR_ACTUAL_API_KEY_HERE"

In [26]:
# Testing placeholder pipeline
query = "What is the policy"
print(f"Querying: {query}")
print("Response:", run_pipeline(query))

Querying: What is the policy


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 28.744453145s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-Fre

In [28]:
# Testing based on Ordinance 03-26
query = "What is the purpose of the Police Advisory Board?"

# Run it
response = run_pipeline(query)
print(f"Query: {query}")
print(f"Result: {response}")

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 55.996126462s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-Fre

In [ ]:
import time
import os
import google.generativeai as genai
from src.llm_engine import MuniciPALEngine

# Forcing a full reset
print("Resetting engine connection...")
del engine  # Deleting the old object
import gc; gc.collect()  # Forcing garbage collection to kill the old gRPC channel

# Re-initialize
# Waiting slightly longer just in case the quota reset is jittery
time.sleep(65) 

#Creating a totally new engine instance
engine = MuniciPALEngine(api_key=os.getenv("GOOGLE_API_KEY"))

# Trying the test again
try:
    response = run_pipeline("What is the purpose of the Police Advisory Board?")
    print("SUCCESS:", response)
except Exception as e:
    print("Still failing:", repr(e))

# realized i hit the api quota limit (>.<) (>W<) :/

Resetting engine connection...
